# Day Feature Engineering: 
## `mart_report_daily_adoption`

This notebook builds the first draft of `mart_report_daily_adoption` from `data/processed/fact_report_views.csv`.

Day objective:
- Aggregate report views to the daily `date` and `report_id` grain
- Create `daily_views`, `unique_viewers`, and `views_per_user`
- Build a per-report daily date spine from each report's launch date to the dataset's global max date
- Fill missing post-launch days with zeros and save the result for later feature engineering and forecasting work


## 1. Setup

We define project paths and import the reusable aggregation helper.


In [6]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.report_features import build_report_daily_adoption

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FACT_REPORT_VIEWS_PATH = DATA_PROCESSED_DIR / "fact_report_views.csv"
OUTPUT_PATH = DATA_PROCESSED_DIR / "mart_report_daily_adoption.csv"


## 2. Load `fact_report_views`

The processed report view fact table already created in the repo.


In [ ]:
if not FACT_REPORT_VIEWS_PATH.exists():
    raise FileNotFoundError(f"Expected input file was not found: {FACT_REPORT_VIEWS_PATH}")

fact_report_views = pd.read_csv(FACT_REPORT_VIEWS_PATH)
print(f"Loaded input from: {FACT_REPORT_VIEWS_PATH}")
print("Shape:", fact_report_views.shape)
display(fact_report_views.head())


## 3. Aggregate to the Daily Report Grain

We first aggregate the event-level fact table to `date` and `report_id`, using `view_count` for total daily views and `user_key` for distinct viewers.


In [8]:
report_daily_base = build_report_daily_adoption(
    fact_report_views=fact_report_views,
    date_col="date_key",
    report_col="report_id",
    user_col="user_key",
    views_col="view_count",
)

display(report_daily_base.head())
print("Shape:", report_daily_base.shape)


,date,report_id,daily_views,unique_viewers,views_per_user
0,2025-01-01,R_001,36,22,1.636364
1,2025-01-02,R_001,41,25,1.640000
2,2025-01-03,R_001,37,27,1.370370
3,2025-01-04,R_001,17,12,1.416667
4,2025-01-05,R_001,30,18,1.666667


Shape: (13627, 5)


## 4. Build the Per-Report Daily Date Spine

Each report starts on its first observed view date. Every report then continues daily until the dataset's global maximum date. Missing post-launch dates are filled with zeros.


In [9]:
report_start_dates = (
    report_daily_base.groupby("report_id", as_index=False)["date"]
    .min()
    .rename(columns={"date": "start_date"})
)
global_max_date = report_daily_base["date"].max()

date_spine_frames = []
for row in report_start_dates.itertuples(index=False):
    report_dates = pd.date_range(start=row.start_date, end=global_max_date, freq="D")
    date_spine_frames.append(
        pd.DataFrame({"date": report_dates, "report_id": row.report_id})
    )

report_date_spine = pd.concat(date_spine_frames, ignore_index=True)

mart_report_daily_adoption = report_date_spine.merge(
    report_daily_base,
    on=["date", "report_id"],
    how="left",
)

mart_report_daily_adoption[["daily_views", "unique_viewers"]] = (
    mart_report_daily_adoption[["daily_views", "unique_viewers"]]
    .fillna(0)
    .astype(int)
)

mart_report_daily_adoption["views_per_user"] = (
    mart_report_daily_adoption["daily_views"]
    .div(mart_report_daily_adoption["unique_viewers"].replace(0, pd.NA))
    .fillna(0.0)
)

mart_report_daily_adoption = mart_report_daily_adoption[
    ["date", "report_id", "daily_views", "unique_viewers", "views_per_user"]
].sort_values(["report_id", "date"]).reset_index(drop=True)

display(mart_report_daily_adoption.head())
print("Shape:", mart_report_daily_adoption.shape)
print("Global max date:", global_max_date.date())


/var/folders/_r/0mbwmsn56rl_rr8h_2gs0tww0000gn/T/ipykernel_41361/2147230076.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0.0)


,date,report_id,daily_views,unique_viewers,views_per_user
0,2025-01-01,R_001,36,22,1.636364
1,2025-01-02,R_001,41,25,1.640000
2,2025-01-03,R_001,37,27,1.370370
3,2025-01-04,R_001,17,12,1.416667
4,2025-01-05,R_001,30,18,1.666667


Shape: (13650, 5)
Global max date: 2026-03-31


## 5. Validation Checks

These checks confirm the final mart has the expected grain, no pre-launch rows, no nulls in key columns, and a consistent end date across reports.


In [10]:
grain_is_unique = not mart_report_daily_adoption.duplicated(subset=["date", "report_id"]).any()
null_counts = mart_report_daily_adoption[
    ["date", "report_id", "daily_views", "unique_viewers", "views_per_user"]
].isnull().sum()

validation_dates = mart_report_daily_adoption.merge(report_start_dates, on="report_id", how="left")
has_prelaunch_rows = (validation_dates["date"] < validation_dates["start_date"]).any()
report_end_dates = mart_report_daily_adoption.groupby("report_id")["date"].max()
all_reports_end_at_global_max = report_end_dates.eq(global_max_date).all()
zero_view_logic_holds = (
    mart_report_daily_adoption.loc[mart_report_daily_adoption["unique_viewers"] == 0, "views_per_user"]
    .eq(0)
    .all()
)

print("One row per date/report_id:", grain_is_unique)
print("No rows before a report's first observed view date:", not has_prelaunch_rows)
print("All reports end at the dataset global max date:", all_reports_end_at_global_max)
print("views_per_user is 0 when unique_viewers is 0:", zero_view_logic_holds)

print("\nNull counts in key columns:")
display(null_counts.to_frame(name="null_count"))


One row per date/report_id: True
No rows before a report's first observed view date: True
All reports end at the dataset global max date: True
views_per_user is 0 when unique_viewers is 0: True

Null counts in key columns:


,null_count
date,0
report_id,0
daily_views,0
unique_viewers,0
views_per_user,0


## 6. Summary Statistics

A quick distribution check helps confirm the Day 1 features look reasonable before saving the mart.


In [11]:
summary_stats = mart_report_daily_adoption[
    ["daily_views", "unique_viewers", "views_per_user"]
].describe()
display(summary_stats)


,daily_views,unique_viewers,views_per_user
count,13650.000000,13650.000000,13650.000000
mean,46.682857,30.110330,1.547944
std,31.419843,20.007464,0.230424
min,0.000000,0.000000,0.000000
25%,24.000000,16.000000,1.424242
50%,41.000000,26.000000,1.538462
75%,62.000000,39.750000,1.666667
max,201.000000,119.000000,4.000000


## 7. Save the Day 1 Output

The completed daily adoption mart is written to the processed data folder for reuse in later feature engineering and forecasting steps.


In [ ]:
mart_report_daily_adoption.to_csv(OUTPUT_PATH, index=False)
print(f"Saved mart_report_daily_adoption to: {OUTPUT_PATH}")
